# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cb-bantong/FlyRank-ML-Program/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type: Ranking / Priority Opportunity Scoring**

**Why this task type:**
In search discoverability and content optimization, human editors and SEO teams operate under strict capacity constraints (e.g., reviewing 20 to 50 URLs per week across an inventory of 30,000+ pages). A binary classification model ("needs review" vs. "healthy") is ineffective because thousands of pages underperform their potential to varying degrees, creating an unprioritized backlog. Unsupervised clustering groups similar pages but does not answer the operational question of which page to open first.

By framing this as a **ranking / scoring task**, we calculate an **opportunity score** based on the deficit between a page's observed CTR and its peer benchmark expected CTR (conditioned on position tier and intent), scaled by impressions. This produces an ordered queue where high-exposure URLs with the greatest actionable traffic upside appear at the top.

In [1]:
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure working directory is the repo root
current_dir = Path.cwd()
while not (current_dir / "data" / "raw").is_dir() and current_dir.parent != current_dir:
    current_dir = current_dir.parent
os.chdir(current_dir)

# Load the anonymized starter dataset
data_path = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

# Demonstrate why classification/thresholding fails and ranking is required:
# A binary rule (e.g. CTR < 0.5%) flags an overwhelming volume of pages
visible_df = df[df["impressions_90d"] >= 100].copy()
flagged_binary = visible_df[visible_df["ctr"] < 0.5]

print(f"Total content items in dataset: {len(df):,}")
print(f"Visible content items (impressions >= 100): {len(visible_df):,}")
print(f"Pages flagged by binary rule (CTR < 0.5%): {len(flagged_binary):,} ({len(flagged_binary) / len(visible_df):.1%} of visible inventory)")
print("Conclusion: Binary classification floods editorial capacity; ranking/scoring produces an actionable priority queue.")

Total content items in dataset: 30,000
Visible content items (impressions >= 100): 22,006
Pages flagged by binary rule (CTR < 0.5%): 18,614 (84.6% of visible inventory)
Conclusion: Binary classification floods editorial capacity; ranking/scoring produces an actionable priority queue.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target Definition:**
The primary continuous target is the **CTR Opportunity Gap (Estimated Lost Clicks)**:
$$\text{CTR Opportunity Gap} = \max(0, \text{Expected CTR}_{\text{tier}} - \text{Observed CTR})$$
$$\text{Estimated Lost Clicks} = \frac{\text{CTR Opportunity Gap}}{100} \times \text{impressions\_90d}$$

For validation audits and top-K precision checks, the binary proxy label is:
$$\text{is\_underperforming\_ctr} = 1 \text{ if (impressions\_90d} \ge 100 \text{ and CTR} \le 25\text{th percentile CTR of its position tier), else } 0$$

**Origin of the label (Observed vs Defined):**
This label is derived strictly from **observed search measurements** (`impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `position_tier`). It intentionally does **not** rely on FlyRank's internal product decision flags (`health_score`, `needs_ctr_fix`, `priority_score`), ensuring our model learns directly from empirical search performance rather than copying existing heuristic decisions (avoiding circular leakage).

In [2]:
# Compute observed peer CTR benchmarks (median and 25th percentile) per position tier
tier_benchmarks = visible_df.groupby("position_tier")["ctr"].agg(
    page_count="count",
    median_ctr="median",
    p25_ctr=lambda x: x.quantile(0.25),
    mean_ctr="mean"
).reset_index()

print("Observed Peer CTR Benchmarks by Position Tier (impressions >= 100):")
print(tier_benchmarks.to_string(index=False))

# Map benchmarks to calculate CTR gap, lost clicks, and proxy label
median_dict = tier_benchmarks.set_index("position_tier")["median_ctr"].to_dict()
p25_dict = tier_benchmarks.set_index("position_tier")["p25_ctr"].to_dict()

visible_df["expected_ctr"] = visible_df["position_tier"].map(median_dict)
visible_df["p25_ctr"] = visible_df["position_tier"].map(p25_dict)
visible_df["ctr_gap"] = (visible_df["expected_ctr"] - visible_df["ctr"]).clip(lower=0)
visible_df["lost_clicks"] = (visible_df["ctr_gap"] / 100.0) * visible_df["impressions_90d"]
visible_df["is_underperforming_ctr"] = (visible_df["ctr"] <= visible_df["p25_ctr"]).astype(int)

print(f"\nProxy Label Base Rate (severe underperforming CTR across visible pages): {visible_df['is_underperforming_ctr'].mean():.2%}")

Observed Peer CTR Benchmarks by Position Tier (impressions >= 100):
position_tier  page_count  median_ctr  p25_ctr  mean_ctr
         deep         879        0.00     0.00  0.062162
       page_1        8633        0.23     0.09  0.354157
     page_3_5        6058        0.06     0.00  0.137882
     striking        5903        0.15     0.00  0.262529
        top_3         533        0.19     0.05  0.334034

Proxy Label Base Rate (severe underperforming CTR across visible pages): 31.76%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Defensible Metric: Precision@50 (and Precision@20) + Lift over Baseline**

**Why this metric:**
Because editorial teams inspect recommendations in finite weekly batches (e.g. 20 to 50 URLs), the primary operational cost is false discovery—recommending pages that have little actual room for improvement or trivial traffic. Global metrics like ROC-AUC or accuracy are diluted by thousands of non-actionable pages. Precision@K measures the exact accuracy of the surfaced review queue.

**What number means 'good':**
- **Base Rate:** 31.8% of visible pages meet the bottom-quartile CTR criterion within their tier.
- **Naive Rule Baseline:** A simple heuristic rule (e.g. Page 1 or Striking position with CTR < 0.5% and impressions >= 500) yields a Precision@50 of ~0.36 to 0.44.
- **Target Bar for 'Good':** **Precision@50 $\ge 0.70$ (a $1.8\times$ to $2.0\times$ lift over the naive baseline rule)**. This ensures that at least 35 of the top 50 surfaced URLs represent verified, high-leverage optimization opportunities.

In [3]:
# Define a naive heuristic baseline: Page 1 / Striking position, impressions >= 500, CTR < 0.5%
visible_df["naive_rule_score"] = (
    (visible_df["position_tier"].isin(["top_3", "page_1", "striking"])).astype(int) * 30 +
    (visible_df["impressions_90d"] >= 500).astype(int) * 30 +
    (visible_df["ctr"] < 0.5).astype(int) * 40
)

# Top 50 by naive heuristic
top50_baseline = visible_df.sort_values(by=["naive_rule_score", "impressions_90d"], ascending=False).head(50)
baseline_prec50 = top50_baseline["is_underperforming_ctr"].mean()

# Top 50 by position-adjusted opportunity ranking (lost clicks)
top50_opp = visible_df.sort_values(by="lost_clicks", ascending=False).head(50)
opp_prec50 = top50_opp["is_underperforming_ctr"].mean()

base_rate = visible_df["is_underperforming_ctr"].mean()
lift = opp_prec50 / max(baseline_prec50, 0.001)

print(f"Base Rate (Random Selection):       {base_rate:.3f} ({base_rate:.1%})")
print(f"Naive Rule Baseline Precision@50:   {baseline_prec50:.3f} ({baseline_prec50:.1%})")
print(f"Opportunity Model Precision@50:     {opp_prec50:.3f} ({opp_prec50:.1%})")
print(f"Observed Lift over Naive Baseline:  {lift:.2f}x")

Base Rate (Random Selection):       0.318 (31.8%)
Naive Rule Baseline Precision@50:   0.360 (36.0%)
Opportunity Model Precision@50:     0.680 (68.0%)
Observed Lift over Naive Baseline:  1.89x


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis (Grain):**
**One row = one pseudonymized content item (page) associated with a pseudonymized client**, observed over a trailing 90-day search and analytics aggregation window.

- **Granular ID:** `content_id` (30,000 unique rows in this slice).
- **Grouping Entity:** `client_id` (32 distinct clients, used for client-holdout validation splits to prevent cross-client data leakage).
- **Lane 4 Signal Vector:** Observed search exposure (`impressions_90d`, `avg_position`, `position_tier`), engagement capture (`clicks_90d`, `ctr`, `sessions_90d`, `engagement_rate`), and content properties (`main_intent`, `content_type`, `word_count`).

In [4]:
# Verify dataframe dimensions and grain
print(f"Dataframe dimensions: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique content_id count: {df['content_id'].nunique():,} (confirms strictly 1 row per unique content item)")
print(f"Unique client_id count:  {df['client_id'].nunique():,} (used for client-holdout grouped splits)")

# Display unit of analysis with Lane 4 feature slice
lane4_cols = [
    "content_id", "client_id", "position_tier", "avg_position",
    "impressions_90d", "clicks_90d", "ctr", "sessions_90d", "engagement_rate"
]
display_df = df[lane4_cols].head(5)
print("\nPreview of Unit of Analysis (first 5 rows):")
display_df

Dataframe dimensions: 30,000 rows × 44 columns
Unique content_id count: 30,000 (confirms strictly 1 row per unique content item)
Unique client_id count:  32 (used for client-holdout grouped splits)

Preview of Unit of Analysis (first 5 rows):
         content_id        client_id position_tier  avg_position  impressions_90d  clicks_90d  ctr  sessions_90d  engagement_rate
content_304f48230142 client_f369cb89fc      striking          10.6             3803          29 0.76            17             5.88
content_573d4fa7df70 client_f369cb89fc        page_1           9.3            14798          12 0.08            11             0.00
content_4fa7df70a6c2 client_f369cb89fc      page_3_5          21.1              157           0 0.00             0             0.00
content_7df70a6c2f90 client_f369cb89fc      page_3_5          28.4              214           0 0.00             0             0.00
content_f70a6c2f901a client_f369cb89fc        page_1           4.1             6428          14 0.2

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A static if-statement (such as `if ctr < 0.5% then flag_for_review`) fails on four structural aspects of real search data:

1. **Non-linear Position Dynamics:** Expected CTR varies exponentially with rank. A CTR of 0.3% is strong for position 18, but a severe failure for position 2. A fixed threshold flags thousands of healthy deep pages while missing top-ranking pages that are leaking massive traffic.
2. **Sample Size & Low-Volume Noise:** With 15 impressions, 0 clicks yields 0.0% CTR, but a single click produces 6.7% CTR. A fixed rule cannot distinguish between statistical noise and true snippet/content problems.
3. **Multivariate Context & Search Intent:** Commercial, transactional, and informational queries exhibit different click behavior curves. A product comparison naturally attracts different click and reading patterns than an informational definition.
4. **Value-Weighted Prioritization:** Fixed rules output boolean flags without quantifying business value. ML models the continuous interaction of position, intent, and volume, estimating exactly how many clicks are being lost to maximize editorial ROI.

In [5]:
# 1. Demonstrate false alarms: deep position pages flagged by flat rule (CTR < 0.5%)
deep_flagged = df[(df["position_tier"] == "deep") & (df["ctr"] < 0.5)]
print(f"1. False alarms from flat rule: {len(deep_flagged):,} deep-position pages flagged (wasting editorial review).")

# 2. Demonstrate low-volume noise:
low_vol = df[df["impressions_90d"] < 50]
zero_click_low_vol = low_vol[low_vol["clicks_90d"] == 0]
print(f"2. Low-volume noise: {len(zero_click_low_vol):,} pages have < 50 impressions and 0 clicks (0.0% CTR by noise, not snippet failure).")

# 3. High-leverage opportunities discovered by continuous opportunity scoring:
top_5_leverage = visible_df.sort_values(by="lost_clicks", ascending=False)[
    ["content_id", "position_tier", "avg_position", "impressions_90d", "clicks_90d", "ctr", "expected_ctr", "lost_clicks"]
].head(5)

print("\nTop 5 High-Leverage Opportunity Pages identified by position-adjusted ranking:")
top_5_leverage

1. False alarms from flat rule: 1,264 deep-position pages flagged (wasting editorial review).
2. Low-volume noise: 3,962 pages have < 50 impressions and 0 clicks (0.0% CTR by noise, not snippet failure).

Top 5 High-Leverage Opportunity Pages identified by position-adjusted ranking:
         content_id position_tier  avg_position  impressions_90d  clicks_90d  ctr  expected_ctr  lost_clicks
content_36ff89c8214e        page_1           7.3           295097         154 0.05          0.23   531.174600
content_c8e9d6ab9013        page_1           9.7           208678           0 0.00          0.23   479.959400
content_5fe46e04994d        page_1           4.2           517715         714 0.14          0.23   465.943500
content_c84a0ab98e90        page_1           7.8           223271          71 0.03          0.23   446.542000
content_8451fc6f034d         top_3           2.3           272144          76 0.03          0.19   435.430400


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.